# 03_GSE156728_qc_selection

**Thesis Methods section(s): 4.1.1, 4.1.2, 4.1.3**

**Reads:** GSE156728 per-cancer CD8 count matrices and the series metadata file (GEO).

**Writes:** GSE156728_CD8_STRICT_postQC_scvi_rawcounts.h5ad

**Notes:** Cells were already CD8-restricted by the original authors; no marker rule applied. Contains an exploratory per-dataset scVI model NOT reported in the thesis. Some cells also read the CD4 counts files, which were inspected but not used in the atlas.

Input data are not included in this repository. Set `DATA_ROOT` below to a local folder
holding the GEO downloads; see `README.md` for accessions and the expected layout.


In [ ]:
# Root folder for input data (NOT included in this repository).
# Set the DATA_ROOT environment variable, or edit the fallback below.
import os
DATA_ROOT = os.environ.get("DATA_ROOT", "data")


In [ ]:
# ----------------------------------
# 🔧 1. Install these once (if not done)
# ----------------------------------
# pip install scanpy scvi-tools anndata torch matplotlib pandas igraph leidenalg

In [ ]:
# 📦 2. Imports
# ----------------------------------
import os
import scanpy as sc
import scvi
import pandas as pd
from scipy.sparse import csr_matrix

In [ ]:
# 📂 3. Function to load .txt and return AnnData
# ----------------------------------
def load_txt_to_anndata(filepath, cancer_type, cell_type):
    df = pd.read_csv(filepath, sep="\t", index_col=0)
    adata = sc.AnnData(X=csr_matrix(df.T))  # cells x genes
    adata.var_names = df.index
    adata.obs_names = df.columns
    adata.obs["cancer_type"] = cancer_type
    adata.obs["cell_type"] = cell_type
    return adata

In [ ]:
# 🧬 4A. Load all datasets (CD8)
# ----------------------------------
BC_CD8 = load_txt_to_anndata(f"{DATA_ROOT}/Single-cell RNA seq/GSE156728/Merged/GSE156728_BC_10X.CD8.counts.txt", "BC", "CD8")
BCL_CD8 = load_txt_to_anndata(f"{DATA_ROOT}/Single-cell RNA seq/GSE156728/Merged/GSE156728_BCL_10X.CD8.counts.txt", "BCL", "CD8")
ESCA_CD8 = load_txt_to_anndata(f"{DATA_ROOT}/Single-cell RNA seq/GSE156728/Merged/GSE156728_ESCA_10X.CD8.counts.txt", "ESCA", "CD8")
MM_CD8 = load_txt_to_anndata(f"{DATA_ROOT}/Single-cell RNA seq/GSE156728/Merged/GSE156728_MM_10X.CD8.counts.txt", "MM", "CD8")
PACA_CD8 = load_txt_to_anndata(f"{DATA_ROOT}/Single-cell RNA seq/GSE156728/Merged/GSE156728_PACA_10X.CD8.counts.txt", "PACA", "CD8")
RC_CD8 = load_txt_to_anndata(f"{DATA_ROOT}/Single-cell RNA seq/GSE156728/Merged/GSE156728_RC_10X.CD8.counts.txt", "RC", "CD8")
THCA_CD8 = load_txt_to_anndata(f"{DATA_ROOT}/Single-cell RNA seq/GSE156728/Merged/GSE156728_THCA_10X.CD8.counts.txt", "THCA", "CD8")
UCEC_CD8 = load_txt_to_anndata(f"{DATA_ROOT}/Single-cell RNA seq/GSE156728/Merged/GSE156728_UCEC_10X.CD8.counts.txt", "UCEC", "CD8")

# 🧬 4B. Load all datasets (CD4)
# ----------------------------------
BC_CD4 = load_txt_to_anndata(f"{DATA_ROOT}/Single-cell RNA seq/GSE156728 CD4/GSE156728_BC_10X.CD4.counts.txt", "BC", "CD4")
BCL_CD4 = load_txt_to_anndata(f"{DATA_ROOT}/Single-cell RNA seq/GSE156728 CD4/GSE156728_BCL_10X.CD4.counts.txt", "BCL", "CD4")
ESCA_CD4 = load_txt_to_anndata(f"{DATA_ROOT}/Single-cell RNA seq/GSE156728 CD4/GSE156728_ESCA_10X.CD4.counts.txt", "ESCA", "CD4")
MM_CD4 = load_txt_to_anndata(f"{DATA_ROOT}/Single-cell RNA seq/GSE156728 CD4/GSE156728_MM_10X.CD4.counts.txt", "MM", "CD4")
PACA_CD4 = load_txt_to_anndata(f"{DATA_ROOT}/Single-cell RNA seq/GSE156728 CD4/GSE156728_PACA_10X.CD4.counts.txt", "PACA", "CD4")
RC_CD4 = load_txt_to_anndata(f"{DATA_ROOT}/Single-cell RNA seq/GSE156728 CD4/GSE156728_RC_10X.CD4.counts.txt", "RC", "CD4")
THCA_CD4 = load_txt_to_anndata(f"{DATA_ROOT}/Single-cell RNA seq/GSE156728 CD4/GSE156728_THCA_10X.CD4.counts.txt", "THCA", "CD4")
UCEC_CD4 = load_txt_to_anndata(f"{DATA_ROOT}/Single-cell RNA seq/GSE156728 CD4/GSE156728_UCEC_10X.CD4.counts.txt", "UCEC", "CD4")

In [ ]:
# 🔗 5. Concatenate all into one AnnData
# ----------------------------------
import anndata as ad

GSE156728_Merged = ad.concat(
    [
        BC_CD8, BCL_CD8, ESCA_CD8, MM_CD8, PACA_CD8, RC_CD8, THCA_CD8, UCEC_CD8,
        BC_CD4, BCL_CD4, ESCA_CD4, MM_CD4, PACA_CD4, RC_CD4, THCA_CD4, UCEC_CD4
    ]
)

In [ ]:
GSE156728_Merged.obs_names.name = "cellID"

In [ ]:
print(GSE156728_Merged.obs.head())

In [ ]:
# Load cell annotations
cell_annotations = pd.read_csv(f"{DATA_ROOT}/Single-cell RNA seq/GSE156728/GSE156728_metadata.txt/GSE156728_metadata.txt", sep='\t', index_col="cellID", header=0)

In [ ]:
# Check alignment
print(cell_annotations.head())

In [ ]:
cell_annotations = cell_annotations.drop(columns=['libraryID'])

In [ ]:
GSE156728_Merged.obs = GSE156728_Merged.obs.drop(columns=["loc"])

In [ ]:
# Optionally rename columns
cell_annotations.columns = ["patient", "libraryID", "loc", "meta.cluster", "platform"]

In [ ]:
# Overlap between barcodes
matching = GSE156728_Merged.obs.index.intersection(cell_annotations.index)
print(f"Matching barcodes: {len(matching)} / {GSE156728_Merged.n_obs}")

In [ ]:
# Join annotations
GSE156728_Merged.obs = GSE156728_Merged.obs.join(cell_annotations, how="left")

In [ ]:
GSE156728_Merged.obs.columns

In [ ]:
New_order = [
    "cancer_type", "cell_type", "cancerType", "meta.cluster", "patient", "libraryID", "loc", "platform"
]

GSE156728_Merged.obs = GSE156728_Merged.obs[New_order]

In [ ]:
# Map sample_type to general tissue source
tissue_map = {
    "T": "Tumor",
    "N": "Normal",
    "P": "Blood"
    }

# Create new column in adata_Merged.obs
GSE156728_Merged.obs["sample_type"] = GSE156728_Merged.obs["loc"].map(tissue_map)

In [ ]:
# Map sample_type to general tissue source
cancer_name = {
    "BC": "Breast Cancer",
    "BCL": "B Cell Lymphoma",
    "MM": "Multiple Myeloma",
    "PACA": "Pancreatic Adenocarcinoma",
    "ESCA": "Esophageal Adenocarcinoma",
    "RC": "Renal Cell Carcinoma",
    "THCA": "Thyroid Carcinoma",
    "UCEC": "Uterine Corpus Endometrial Carcinoma"
    }

# Create new column in adata_Merged.obs
GSE156728_Merged.obs["cancer_type"] = GSE156728_Merged.obs["cancer_type"].map(cancer_name)

In [ ]:
GSE156728_Merged.obs["sample_type"].value_counts()

In [ ]:
GSE156728_Merged.obs["cancer_type"].value_counts()

In [ ]:
GSE156728_Merged.obs["patient"].value_counts()

In [ ]:
GSE156728_Merged.obs["cell_type"].value_counts()

In [ ]:
# 📐 Count cells per patient and sample_type and cancer_type
counts = GSE156728_Merged.obs.groupby(["patient", "sample_type", "cancer_type", "cell_type"]).size().reset_index(name="cell_count")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(20, 6))

# Sort patients by patient (since counts does not have cancer_type column)
df_sorted = counts.sort_values(by=['patient'])

sns.barplot(
    data=df_sorted,
    x='patient',
    y='cell_count',
    hue='sample_type',
    palette={'Tumor': '#2b4f81', 'Normal': '#6fa2d4', 'Blood': '#5ecb98'},
    ci=None # Disable confidence intervals for clarity
)

plt.xticks(rotation=90, ha='center')
plt.xlabel("Patient")
plt.ylabel("Number of Cells")
plt.title("Cell Counts per Patient and Sample Type")
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(20, 6))

# Sort patients by patient (since counts does not have cancer_type column)
df_sorted = counts.sort_values(by=['cancer_type'])

sns.barplot(
    data=df_sorted,
    x='cancer_type',
    y='cell_count',
    hue='sample_type',
    ci=None,
    palette={'Tumor': '#2b4f81', 'Normal': '#6fa2d4', 'Blood': '#5ecb98'}
)

plt.xticks(rotation=0, ha='center')
plt.xlabel("Cancer Type")
plt.ylabel("Number of Cells")
plt.title("Cell Counts per Cancer and Sample Type")
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(20, 6))

# Sort patients by patient (since counts does not have cancer_type column)
df_sorted = counts.sort_values(by=['cancer_type', 'cell_type'])

sns.barplot(
    data=df_sorted,
    x='cancer_type',
    y='cell_count',
    hue='cell_type',
    errorbar=None,
    palette={'CD8': "#b86b9a", 'CD4': "#6566B4"}
)

plt.xticks(rotation=0, ha='center')
plt.xlabel("Cancer Type")
plt.ylabel("Number of Cells")
plt.title("Cell Counts per Cancer and Sample Type")
plt.tight_layout()
plt.show()

In [ ]:
# Quality Control (QC)
import scanpy as sc
import matplotlib.pyplot as plt

# Calculate QC metrics
GSE156728_Merged.var['mt'] = GSE156728_Merged.var_names.str.upper().str.startswith('MT-')  # Ensures uppercase compatibility
sc.pp.calculate_qc_metrics(GSE156728_Merged, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

# Plot QC metrics before filtering
fig, axs = plt.subplots(1, 3, figsize=(15, 4))

# Total counts per cell
axs[0].hist(GSE156728_Merged.obs['total_counts'], bins=50, color='skyblue')
axs[0].set_title('Total Counts per Cell')
axs[0].set_xlabel('Total Counts')
axs[0].set_ylabel('Number of Cells')

# Number of genes per cell
axs[1].hist(GSE156728_Merged.obs['n_genes_by_counts'], bins=50, color='lightgreen')
axs[1].set_title('Number of Genes per Cell')
axs[1].set_xlabel('Genes')
axs[1].set_ylabel('Number of Cells')

# Percent mitochondrial genes
axs[2].hist(GSE156728_Merged.obs['pct_counts_mt'], bins=50, color='salmon')
axs[2].set_title('% Mitochondrial Genes')
axs[2].set_xlabel('% MT Genes')
axs[2].set_ylabel('Number of Cells')

plt.tight_layout()
plt.show()

In [ ]:
# You can now visually decide thresholds.
# Here's one possible filtering step based on common practice:
sc.pp.filter_cells(GSE156728_Merged, min_genes=200)
sc.pp.filter_genes(GSE156728_Merged, min_cells=3)
GSE156728_Merged = GSE156728_Merged[GSE156728_Merged.obs.pct_counts_mt < 10, :]  # Modify threshold based on histogram

In [ ]:
# 📌 Save raw version for scVI
GSE156728_raw_scVI = GSE156728_Merged.copy()

In [ ]:
# 📍 UMAP before scVI (PCA-based)
# ----------------------------------
sc.pp.normalize_total(GSE156728_Merged, target_sum=1e4)
sc.pp.log1p(GSE156728_Merged)
sc.pp.highly_variable_genes(GSE156728_Merged, n_top_genes=2000, subset=True)
sc.pp.scale(GSE156728_Merged, max_value=10)
sc.tl.pca(GSE156728_Merged, svd_solver='arpack')

In [ ]:
# 🔍 Elbow plot to choose number of PCs
import matplotlib.pyplot as plt
import numpy as np

explained = GSE156728_Merged.uns["pca"]["variance_ratio"]
plt.figure(figsize=(6, 4))
plt.plot(range(1, len(explained) + 1), explained, marker='o')
plt.xlabel('Principal Component')
plt.ylabel('Explained Variance Ratio')
plt.title('PCA Elbow Plot')
plt.grid(True)
plt.show()

In [ ]:
# Dimensionality reduction & clustering
sc.pp.neighbors(GSE156728_Merged, n_neighbors=15, n_pcs=20)
sc.tl.umap(GSE156728_Merged)
sc.tl.leiden(GSE156728_Merged, resolution=0.5)

In [ ]:
# UMAP plot (PCA-based)
sc.pl.umap(
    GSE156728_Merged,
    color=["leiden"],
    title=["Before scVI: Clusters"],
)

In [ ]:
# UMAP plot (PCA-based)
sc.pl.umap(
    GSE156728_Merged,
    color=["cancer_type"],
    title=["Before scVI: Cancer Type"],
)

In [ ]:
# UMAP plot (PCA-based)
sc.pl.umap(
    GSE156728_Merged,
    color=["cell_type"],
    title=["Before scVI: Cell Type"],
)

In [ ]:
# UMAP plot (PCA-based)
sc.pl.umap(
    GSE156728_Merged,
    color=["patient"],
    title=["Before scVI: Patient"],
)

In [ ]:
# UMAP plot (PCA-based)
sc.pl.umap(
    GSE156728_Merged,
    color=["sample_type"],
    title=["Before scVI: Sample Type"],
)

In [ ]:
# Optionally save the file
GSE156728_Merged.write(
    f"{DATA_ROOT}/scVI/GSE156728_Merged_PCA_based.h5ad"
)

In [ ]:
# Optionally save the file
GSE156728_raw_scVI.write(
    f"{DATA_ROOT}/scVI/GSE156728_raw.h5ad"
)

In [ ]:
# Load the saved AnnData object
import scanpy as sc
GSE156728_Merged = sc.read(f"{DATA_ROOT}/scVI/GSE156728_Merged_umap_before_scvi.h5ad")
GSE156728_raw_scVI = sc.read(f"{DATA_ROOT}/scVI/GSE156728_raw_before_scvi.h5ad")

In [ ]:
import os
import scvi

# Define save path
model_path = "scvi_model/GSE156728_scvi_model"

# Make sure the folder exists
os.makedirs("scvi_model", exist_ok=True)

# Set up AnnData for scVI
scvi.model.SCVI.setup_anndata(GSE156728_raw_scVI, batch_key="patient")

# Load or train
if os.path.exists(os.path.join(model_path, "model.pt")):
    model = scvi.model.SCVI.load(model_path, GSE156728_raw_scVI)
    print("Loaded saved scVI model.")
else:
    model = scvi.model.SCVI(GSE156728_raw_scVI)
    model.train(max_epochs=50)
    model.save(model_path, overwrite=True)
    print("Trained and saved scVI model.")

In [ ]:
import os
print(os.getcwd())

In [ ]:
# Change working directory
import os

# Change working directory
os.chdir(f"{DATA_ROOT}/scVI")

# Verify change
print(os.getcwd())

In [ ]:
# Plot scVI Training Loss Curve
# ----------------------------------
import matplotlib.pyplot as plt

# Get history directly from trained model
history = model.history
print("Available keys in history:", history.keys())
elbo = history["elbo_train"]

# Plot
plt.plot(range(1, len(elbo) + 1), elbo, marker='o')
plt.xlabel("Epoch")
plt.ylabel("Training ELBO Loss")
plt.title("scVI Training Loss")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# 📉 Get scVI latent space and compute UMAP
GSE156728_raw_scVI.obsm["X_scVI"] = model.get_latent_representation()

# Use scVI latent space for graph and visualization
sc.pp.neighbors(GSE156728_raw_scVI, use_rep="X_scVI")
sc.tl.umap(GSE156728_raw_scVI)

Export the scVI-processed object for Seurat: expression matrix, cell metadata, scVI latent space and UMAP coordinates.

In [ ]:
# Optionally save the file
GSE156728_raw_scVI.write(
    f"{DATA_ROOT}/scVI/GSE156728_scVI_based.h5ad"
)

In [ ]:
# Load the saved AnnData object
GSE156728_scVI_based = sc.read(f"{DATA_ROOT}/scVI/3 GSE156728/3C GSE156728_scVI_based.h5ad")

In [ ]:
import scanpy as sc
import pandas as pd
import os

adata = sc.read("GSE156728_raw_after_scVI.h5ad")
output_dir = "GSE156728_for_Seurat"
os.makedirs(output_dir, exist_ok=True)

# Export expression matrix (dense)
expr_df = pd.DataFrame(
    adata.X.toarray() if hasattr(adata.X, "toarray") else adata.X,
    index=adata.obs_names,
    columns=adata.var_names
)
expr_df.to_csv(f"{output_dir}/counts_matrix.csv")

# Export metadata
adata.obs.to_csv(f"{output_dir}/metadata.csv")

# Export latent space (scVI)
if "X_scVI" in adata.obsm:
    pd.DataFrame(adata.obsm["X_scVI"], index=adata.obs_names).to_csv(f"{output_dir}/scVI_latent.csv")

# Export UMAP
if "X_umap" in adata.obsm:
    pd.DataFrame(adata.obsm["X_umap"], index=adata.obs_names).to_csv(f"{output_dir}/umap.csv")

In [ ]:
# UMAP & Clustering on scVI Latent Space
# --------------------------------------------

# Step 1: Compute neighborhood graph using the scVI latent space
sc.pp.neighbors(GSE156728_raw_scVI, use_rep="X_scVI", n_neighbors=15)

# Step 2: Run UMAP (2D)
sc.tl.umap(GSE156728_raw_scVI, n_components=2)

# Step 3: Leiden clustering
sc.tl.leiden(GSE156728_raw_scVI, resolution=0.5)

# Step 4 (Optional): Check number of clusters
n_clusters = GSE156728_raw_scVI.obs["leiden"].nunique()
print(f"✅ Leiden clustering at resolution 0.5 → {n_clusters} clusters")


In [ ]:
# UMAP plot (scVI-based)
sc.pl.umap(
    GSE156728_raw_scVI,
    color=["leiden"],
    title=["After scVI: Clusters"],
)

In [ ]:
# UMAP plot (scVI-based)
sc.pl.umap(
    GSE156728_raw_scVI,
    color=["cancer_type"],
    title=["After scVI: Cancer Type"],
)

In [ ]:
# UMAP plot (scVI-based)
sc.pl.umap(
    GSE156728_raw_scVI,
    color=["cell_type"],
    title=["After scVI: Cell Type"],
)

In [ ]:
# UMAP plot (scVI-based)
sc.pl.umap(
    GSE156728_raw_scVI,
    color=["patient"],
    title=["After scVI: Patient"],
)

In [ ]:
# UMAP plot
sc.pl.umap(
    GSE156728_raw_scVI,
    color=["sample_type"],
    title=["After scVI: Sample Type"],
)

In [ ]:
# Load the saved AnnData object
GSE156728_scVI_based = sc.read(f"{DATA_ROOT}/scVI/3 GSE156728/3C GSE156728_scVI_based.h5ad")

In [ ]:
# Dataset subset: GSE108989
GSE156728_scVI_based = GSE156728_scVI_based[GSE156728_scVI_based.obs["cell_type"] == "CD8", :]  

In [ ]:
GSE156728_scVI_based.obs.columns

In [ ]:
GSE156728_scVI_based.obs["sample_type"].value_counts()

In [ ]:
# Rename 'old_col' -> 'new_col'
GSE156728_scVI_based.obs.rename(columns={'sample_type': 'tissue_type'}, inplace=True)

In [ ]:
# Eliminate metadata columns that are not needed


In [ ]:
# Load necessary libraries
from scipy.io import mmwrite
from scipy.sparse import csr_matrix
import scanpy as sc
import pandas as pd
import numpy as np

adata = GSE156728_scVI_based.copy()

In [ ]:
# 1) raw counts in .X
if "counts" in adata.layers:
    adata.X = adata.layers["counts"]

In [ ]:
# 2) basic clean-up
sc.pp.filter_genes(adata, min_counts=1)
adata.obs_names_make_unique()
adata.var_names_make_unique()
adata.var_names = adata.var_names.str.replace("_","-", regex=False)

In [ ]:
# 3) compact storage
adata.X = csr_matrix(adata.X).astype(np.int32)

In [ ]:
# --- Sanity: shapes you'll export ---
n_cells, n_genes = adata.n_obs, adata.n_vars
print(f"Cells: {n_cells:,}   Genes: {n_genes:,}")

In [ ]:
# 4) Write MTX **transposed** = genes x cells (what Seurat expects)
mmwrite("counts.mtx", adata.X.T)   # <-- key change

In [ ]:
# 5) Two-column features (ID, name) – here both from var_names
pd.DataFrame({"gene_id": adata.var_names, "gene_name": adata.var_names}).to_csv(
    "features.tsv", sep="\t", header=False, index=False
)

In [ ]:
# 6) One-column barcodes (cells)
pd.DataFrame({"cell": adata.obs_names}).to_csv(
    "barcodes.tsv", sep="\t", header=False, index=False
)

In [ ]:
# 7) Metadata (strings to be safe)
keep = ['cancer_type', 'cell_type', 'cancerType', 'meta.cluster', 'patient',
       'libraryID', 'loc', 'platform', 'tissue_type', 'n_genes_by_counts',
       'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes',
       '_scvi_batch', '_scvi_labels', 'leiden']
keep = [c for c in keep if c in adata.obs.columns]
meta = adata.obs[keep].copy()
for c in meta.columns:
    meta[c] = meta[c].astype(str).str.strip()
meta.to_csv("metadata.csv")

print("✅ Wrote counts.mtx (genes×cells) / features.tsv (2 cols) / barcodes.tsv / metadata.csv")

Export embeddings

In [ ]:
import pandas as pd

# make sure both exist
print(list(adata.obsm.keys()))  # should include "X_scVI" and maybe "X_umap"

pd.DataFrame(adata.obsm["X_scVI"], index=adata.obs_names).to_csv("scvi_embedding.csv")
if "X_umap" in adata.obsm:
    pd.DataFrame(adata.obsm["X_umap"], index=adata.obs_names).to_csv("umap_coords.csv")
print("✅ wrote scvi_embedding.csv and (optionally) umap_coords.csv")